# 🚀 Nova Core - Multi-Dataset Training on Colab

Train **ONE model** on **MULTIPLE Hugging Face datasets** sequentially.

This notebook:
1. Installs Rust + Nova Core
2. Downloads datasets from Hugging Face (direct HTTP - no `datasets` library needed)
3. Trains ONE model on ALL datasets sequentially
4. Saves the final trained model

**No separate models for each dataset - single model, multiple datasets!**

In [ ]:
# @title 1. Install Rust
import sys
import subprocess
import json
import urllib.request
import urllib.error

print("🔧 Installing Rust...")
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y -q 2>&1

# Use %env to set PATH for ALL subsequent ! commands (this is the key fix!)
%env PATH=/root/.cargo/bin:$PATH

print("✅ Rust installed!")
!rustc --version
!cargo --version

In [ ]:
# @title 2. Clone/Build Nova Core
import os
import time

print("📦 Building Nova Core...")

# Check if we're in the right directory
if not os.path.exists('Cargo.toml'):
    print("Cloning Nova Core...")
    !git clone https://github.com/anupbth1/nova_core_pro_V1.git
    %cd nova_core_pro_V1

# Build release (optimized) - this takes ~2-3 minutes
print("⏳ Building release binary (this takes 2-3 minutes)...")
start = time.time()
!cargo build --release 2>&1
elapsed = time.time() - start
print(f"✅ Nova Core built in {elapsed:.0f}s!")
!./target/release/nova --help

In [ ]:
# @title 3. Train ONE model on MULTIPLE datasets
import time

# ===== CONFIGURATION =====
# ⚡ FAST MODE (recommended for testing): dim=64, max_rows=300
# 🚀 FULL MODE: dim=512, max_rows=1000 (takes longer but better results)
DATASETS = "imdb,wikitext,tiny_shakespeare"  # @param {type:"string"}
MAX_ROWS = 300  # @param {type:"integer"}
DIM = 64  # @param {type:"integer"}
CORES = 5  # @param {type:"integer"}
MODEL_NAME = "nova-multi-dataset-v1"  # @param {type:"string"}
PRO_MODE = False  # @param {type:"boolean"}
SPLIT = "train"  # @param ["train", "test", "validation"]

# ===== TRAINING =====
print("=" * 60)
print("🚀 NOVA CORE - MULTI-DATASET TRAINING")
print("=" * 60)
print(f"📋 Datasets: {DATASETS}")
print(f"📊 Max rows per dataset: {MAX_ROWS}")
print(f"🔧 Dim: {DIM}, Cores: {CORES}")
print(f"💾 Model name: {MODEL_NAME}")
print(f"🔥 Pro mode: {PRO_MODE}")
print(f"📂 Split: {SPLIT}")
print("=" * 60)

# Build the command
cmd = f"./target/release/nova multi-hf-train"
cmd += f" --datasets \"{DATASETS}\""
cmd += f" --max-rows {MAX_ROWS}"
cmd += f" --dim {DIM}"
cmd += f" --cores {CORES}"
cmd += f" --model-name \"{MODEL_NAME}\""
cmd += f" --split {SPLIT}"
if PRO_MODE:
    cmd += " --pro"

print(f"\n🔨 Running: {cmd}\n")
start = time.time()
!{cmd}
elapsed = time.time() - start
print(f"\n⏱️ Total training time: {elapsed:.0f}s ({elapsed/60:.1f} min)")

In [ ]:
# @title 4. Test the trained model
MODEL_NAME = "nova-multi-dataset-v1"  # @param {type:"string"}
DIM = 64  # @param {type:"integer"}
CORES = 5  # @param {type:"integer"}

print("🧪 Testing the trained model...")
print("=" * 60)

# Test prompts
test_prompts = [
    "hello",
    "what is your name",
    "how are you",
    "tell me something",
    "the sky is",
]

for prompt in test_prompts:
    cmd = f'./target/release/nova run --input "{prompt}" --model {MODEL_NAME} --dim {DIM} --cores {CORES}'
    print(f"\n📝 Input: {prompt}")
    !{cmd}
    print("-" * 40)

In [ ]:
# @title 5. Interactive Chat with trained model
MODEL_NAME = "nova-multi-dataset-v1"  # @param {type:"string"}
DIM = 64  # @param {type:"integer"}
CORES = 5  # @param {type:"integer"}

print("💬 Starting interactive chat...")
print("Type 'exit' to quit")
print("=" * 60)

!./target/release/nova smart-chat --model {MODEL_NAME} --dim {DIM} --cores {CORES}

In [ ]:
# @title 6. (Optional) Upload model to Hugging Face Hub
MODEL_NAME = "nova-multi-dataset-v1"  # @param {type:"string"}
HF_REPO = "your-username/nova-model"  # @param {type:"string"}
HF_TOKEN = ""  # @param {type:"string"}

if HF_TOKEN and HF_REPO:
    print(f"📤 Uploading model '{MODEL_NAME}' to {HF_REPO}...")
    !./target/release/nova model upload --name {MODEL_NAME} --repo {HF_REPO} --token {HF_TOKEN}
    print("✅ Upload complete!")
else:
    print("⏭️ Skipping upload (set HF_REPO and HF_TOKEN to enable)")

---

## 📋 Available Commands

### Multi-Dataset Training
```bash
nova multi-hf-train \
  --datasets "imdb,wikitext,tiny_shakespeare" \
  --max-rows 300 \
  --dim 64 \
  --cores 5 \
  --model-name "my-universal-model" \
  --split train \
  --pro
```

### Single Dataset Training
```bash
nova hf-train \
  --dataset imdb \
  --max-rows 500 \
  --dim 64 \
  --cores 5 \
  --model-name "my-model"
```

### Test Model
```bash
nova run --input "hello world" --model my-model
```

### Interactive Chat
```bash
nova smart-chat --model my-model
```

### List Models
```bash
nova model list
```

### Upload to Hugging Face
```bash
nova model upload --name my-model --repo username/repo-name --token hf_...
```